# Voyager 1 & 2 — heliocentric trajectories

Voyager 1号・2号を打ち上げ直後から実行日（日本時間の日付）まで、太陽中心・J2000黄道面の3D表示で比較します。単位はAU。惑星8個と2機だけを表示し、スライダーで日付を変更できます。最後のセルでMP4を保存します。

- Voyager 2: 1977-08-20打ち上げ、Horizons収録開始 **1977-08-20 15:32:32.183 TDB**。
- Voyager 1: 1977-09-05打ち上げ、Horizons収録開始 **1977-09-05 13:59:24.383 TDB**。
- 打ち上げから収録開始までの約1時間は補完しません。収録前の探査機は非表示です。
- Horizonsの日次位置を共通時刻へ線形補間します。日付表示はTDB、終了日は実行日の00:00 TDBです。接近時の精密解析用ではありません。
- 惑星は惑星系重心（Horizons ID 1–8）を使用します。点の大きさは視認性のための表示サイズです。
- 初期軌道はHorizonsの近似再構成、近年の軌道は追跡データに基づく予測です。

出典: [NASA mission overview](https://science.nasa.gov/mission/voyager/mission-overview/), [JPL Horizons](https://ssd.jpl.nasa.gov/horizons/)。

In [ ]:
using Pkg
project_dir = isfile(joinpath(pwd(), "Project.toml")) ? pwd() : dirname(pwd())
Pkg.activate(project_dir)
using Horizons3DViewer2
using EphemerisSources
using Dates
using GLMakie
using ProgressMeter
using Serialization
using Base64

In [ ]:
# Settings
end_date = Date(now(UTC) + Hour(9))  # Current calendar date in Japan
stop_time = DateTime(end_date)       # Midnight TDB of that date
sample_step = "1 day"
sampling_interval = 30              # Days per movie frame (about 20 seconds total)
framerate = 30
export_animation = true
refresh_cache = false
AU_KM = 149_597_870.7

# Round the coverage boundaries upward by 1 ms to stay inside the SPK interval.
voyagers = [
    (name="Voyager 1", id=-31, color=:lime,
     first=DateTime(1977, 9, 5, 13, 59, 24, 384)),
    (name="Voyager 2", id=-32, color=:orange,
     first=DateTime(1977, 8, 20, 15, 32, 32, 184)),
]
planets = [
    (name="Mercury", id=1, color=:gray70, period=88),
    (name="Venus",   id=2, color=:wheat, period=225),
    (name="Earth",   id=3, color=:deepskyblue, period=366),
    (name="Mars",    id=4, color=:salmon, period=687),
    (name="Jupiter", id=5, color=:gold, period=4333),
    (name="Saturn",  id=6, color=:khaki, period=10759),
    (name="Uranus",  id=7, color=:turquoise, period=30687),
    (name="Neptune", id=8, color=:cornflowerblue, period=60190),
]
start_time = minimum(v.first for v in voyagers)
@assert stop_time > maximum(v.first for v in voyagers)
# Include both coverage starts and the exact end, even when frames are skipped.
times = sort!(unique!(vcat(collect(start_time:Day(1):stop_time),
                           [v.first for v in voyagers], [stop_time])))
jd = Dates.datetime2julian.(times)
N = length(times)
frames = sort!(unique!(vcat(collect(1:sampling_interval:N),
    [searchsortedfirst(times, v.first) for v in voyagers], [N])))
cache_dir = joinpath(project_dir, "output", "voyager_cache")
mkpath(cache_dir)
movie_path = joinpath(project_dir, "output", "voyager_1_2_$(end_date).mp4")

In [ ]:
# Cache each Horizons response; changing the dates/step changes the cache key.
function fetch_track(body, first_time)
    key = "$(body.id)_$(Dates.format(first_time, "yyyymmdd_HHMMSS"))_$(end_date)_$(replace(sample_step, " "=>"_"))_sun_ecliptic_j2000_kms.jls"
    path = joinpath(cache_dir, key)
    if !refresh_cache && isfile(path)
        return deserialize(path)
    end
    @info "Fetching Horizons ephemeris" target=body.name
    # The range API defaults to ECLIPTIC; an extra day brackets the end for interpolation.
    h = ephemeris(body.id, first_time, stop_time + Day(1), sample_step;
                  wrt="sun", units="KM-S", frame="J2000")
    track = (jd=Float64.(h.t), xyz=horizons_to_matrix(h)[1:3, :] ./ AU_KM)
    @assert length(track.jd) >= 2 && all(diff(track.jd) .> 0)
    @assert all(isfinite, track.xyz)
    serialize(path, track)
    return track
end

# Align all bodies to one time grid. Never extrapolate spacecraft before launch.
function align_track(track, target_jd; first_jd=first(track.jd))
    xyz = fill(NaN, 3, length(target_jd))
    for (i, t) in enumerate(target_jd)
        t < first_jd && continue
        # Allow sub-millisecond rounding in Horizons Julian-date serialization.
        @assert first(track.jd)-1e-8 <= t <= last(track.jd)+1e-8 "Ephemeris does not cover requested epoch"
        t = clamp(t, first(track.jd), last(track.jd))
        k = min(searchsortedlast(track.jd, t), length(track.jd)-1)
        w = (t-track.jd[k]) / (track.jd[k+1]-track.jd[k])
        xyz[:, i] = (1-w) .* track.xyz[:, k] .+ w .* track.xyz[:, k+1]
    end
    return xyz
end

positions = Dict{Int, Matrix{Float64}}()
for body in planets
    positions[body.id] = align_track(fetch_track(body, DateTime(Date(start_time))), jd)
end
for body in voyagers
    positions[body.id] = align_track(fetch_track(body, body.first), jd;
                                    first_jd=Dates.datetime2julian(body.first))
end
println("Loaded $(N) epochs; $(length(frames)) movie frames; $(round(length(frames)/framerate; digits=1)) seconds.")

In [ ]:
# One Sun-centered 3D scene; slider and recording share the same frame index.
set_theme!(theme_black())
fig = Figure(size=(1200, 850), fontsize=16)
frame_index = Observable(1)
extent = 1.12 * maximum(maximum(abs, positions[v.id][:, end]) for v in voyagers)
ax = Axis3(fig[1, 1]; title="Voyager 1 & 2 · Sun-centered · J2000 ecliptic",
    xlabel="x [AU]", ylabel="y [AU]", zlabel="z [AU]",
    aspect=:data, azimuth=-0.3pi, elevation=0.22pi,
    limits=(-extent, extent, -extent, extent, -extent, extent))
scatter!(ax, [Point3f(0, 0, 0)]; color=:yellow, markersize=16, label="Sun")
text!(ax, [Point3f(0, 0, 0)]; text=["Sun"], offset=(8, 8), color=:yellow)

# Planet trails show at most one orbital period; spacecraft retain their full history.
for body in planets
    xyz = positions[body.id]
    trail = @lift begin
        i = $frame_index
        j = searchsortedfirst(jd, jd[i] - body.period)
        [Point3f(xyz[:, k]...) for k in j:i]
    end
    head = @lift [Point3f(xyz[:, $frame_index]...)]
    lines!(ax, trail; color=(body.color, 0.45), linewidth=1)
    scatter!(ax, head; color=body.color, markersize=9, label=body.name)
end
for body in voyagers
    xyz = positions[body.id]
    trail = @lift [Point3f(xyz[:, k]...) for k in 1:$frame_index]
    head = @lift [Point3f(xyz[:, $frame_index]...)]
    lines!(ax, trail; color=body.color, linewidth=2.5, label=body.name)
    scatter!(ax, head; color=body.color, markersize=12)
    text!(ax, head; text=[body.name], color=body.color, offset=(10, 8))
end
Legend(fig[1, 2], ax; framevisible=false)
Label(fig[2, 1:2], @lift(Dates.format(times[$frame_index], "yyyy-mm-dd HH:MM") * " TDB"))
slider = Slider(fig[3, 1:2]; range=1:N, startvalue=1)
on(slider.value) do i
    frame_index[] = i
end
fig

In [ ]:
# Export a synchronized animation, including the final epoch.
if export_animation
    p = Progress(length(frames); desc="Rendering Voyager movie")
    record(fig, movie_path, frames; framerate=framerate) do i
        set_close_to!(slider, i)
        next!(p)
    end
    println("Saved: ", movie_path)
    if isdefined(Main, :IJulia)
        # Embed the movie so it can be played directly in Jupyter.
        video = base64encode(read(movie_path))
        display(MIME"text/html"(), """<video controls loop style="max-width:100%" src="data:video/mp4;base64,$video"></video>""")
    end
end